## **Medical Image Processing**
### Retinal Vessel Challenge

Group BC13:
* S345699 - Sebastiano Natali
* S345744 - Chiara Memoli
* S349049 - Chiara Guadagno
* S349089 - Fabio Galante

In [1]:
# Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
# DEFINE HERE ALL THE NEEDED DIRECTORIES

# Directory that contains all the data/script
current_dir = os.path.join("/content/drive/MyDrive", 'MIP - project')

checkpoint_dir = os.path.join(current_dir, 'BC13 - Submission')  # checkpoint directory
# epoch_number = ..  --> NOT needed (the model is loaded as "best_model.pt")


# Define paths for input images and output masks
test_images_dir = os.path.join(current_dir, 'data', 'test', 'image')
output_masks_dir = os.path.join(checkpoint_dir, 'submission_output')

# 1. Install useful libraries and U-Net definition


In [3]:
import os
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from tqdm import tqdm

In [4]:
# Show versioning of deep learning libraries
import torch, torchvision
print(torch.__version__, torch.cuda.is_available())

2.9.0+cu126 True


In [5]:
import torch.nn as nn
import torch.nn.functional as F

class DoubleConv(nn.Module):
    """Applies two consecutive conv-batchnorm-relu layers"""
    def __init__(self, in_channels, out_channels):
        super(DoubleConv, self).__init__()
        self.double_conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),

            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.double_conv(x)

class UNet(nn.Module):
    def __init__(self,
                 in_channels=1,
                 out_channels=1,
                 init_filters=64,
                 depth=5,
                 bilinear=True):
        super(UNet, self).__init__()
        self.depth = depth
        self.down_layers = nn.ModuleList()
        self.up_layers = nn.ModuleList()
        self.pool = nn.MaxPool2d(2)

        # Encoder
        filters = init_filters
        for d in range(depth):
            conv = DoubleConv(in_channels, filters)
            self.down_layers.append(conv)
            in_channels = filters
            filters *= 2

        # Bottleneck
        self.bottleneck = DoubleConv(in_channels, filters)

        # Decoder
        for d in range(depth):
            filters //= 2
            if bilinear:
                up = nn.Sequential(
                    nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True),
                    nn.Conv2d(filters * 2, filters, kernel_size=1)
                )
            else:
                up = nn.ConvTranspose2d(filters * 2, filters, kernel_size=2, stride=2)
            self.up_layers.append(nn.ModuleDict({
                'up': up,
                'conv': DoubleConv(filters * 2, filters)
            }))

        # Output layer
        self.out_conv = nn.Conv2d(init_filters, out_channels, kernel_size=1)

    def forward(self, x):
        skip_connections = []
        for down in self.down_layers:
            x = down(x)
            skip_connections.append(x)
            x = self.pool(x)

        x = self.bottleneck(x)

        for i in range(self.depth):
            skip = skip_connections[-(i+1)]
            up = self.up_layers[i]['up'](x)
            if up.size() != skip.size():
                # Resize in case of odd size mismatch
                up = F.interpolate(up, size=skip.shape[2:])
            x = torch.cat([skip, up], dim=1)
            x = self.up_layers[i]['conv'](x)

        return self.out_conv(x)

# 2. Load the configuration and weights of the trained U-Net model

In [6]:
import json
import torch

def load_model_from_checkpoint(checkpoint_dir):
    # Path to the JSON file with saved parameters
    params_path = os.path.join(checkpoint_dir, 'training_params.json')

    # Load parameters from JSON
    with open(params_path, 'r') as f:
        params = json.load(f)

    print("Loaded parameters:", params)

    # Create the model using the loaded parameters
    model = UNet(
        in_channels=params['in_channels'],
        out_channels=params['out_channels'],
        init_filters=params['init_filters'],
        depth=params['depth']
    )

    # Path to the checkpoint
    checkpoint_path = os.path.join(checkpoint_dir, f"best_model.pt")

    # Load model state
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    # Load the entire checkpoint dictionary
    checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
    # Extract the model_state_dict from the loaded checkpoint
    model.load_state_dict(checkpoint['model_state_dict'])
    print(f"Checkpoint loaded from {checkpoint_path}")

    return model, params

In [9]:
model, params = load_model_from_checkpoint(checkpoint_dir)
input_size = tuple(params["input_size"])

Loaded parameters: {'input_size': [512, 512], 'in_channels': 3, 'out_channels': 1, 'init_filters': 32, 'depth': 4, 'n_epochs': 75, 'batch_size': 8, 'learning_rate': 0.001, 'checkpoint_freq': 1}
Checkpoint loaded from /content/drive/MyDrive/MIP - project/BC13 - Submission/best_model.pt


# 3. Apply trained model to the test set




In [10]:
from torchvision import transforms
import numpy as np
import skimage.morphology as morph
import torch

def standardize(image):
    # Works for RGB, CIELAB, or Green channel
    # Convert PIL Image to numpy array before standardization
    img = np.array(image).astype(np.float32)
    return (img - np.mean(img)) / (np.std(img) + 1e-7)

# Create output folders if not present
os.makedirs(output_masks_dir, exist_ok=True)

# Send model to device and set to eval
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
model.eval()

# List test image files (assumes .png/.jpg/.jpeg)
image_files = [f for f in os.listdir(test_images_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]

for img_name in image_files:
    img_path = os.path.join(test_images_dir, img_name)

    # Load image
    image = Image.open(img_path).convert('RGB')

    # Apply all the preprocessing
    image = standardize(image)

    # Transpose manually and add batch dimension, then move to device
    image = torch.from_numpy(image.transpose(2, 0, 1) if image.ndim == 3 else image[None, :, :]).unsqueeze(0).to(device)

    # Inference
    with torch.no_grad():
        output = model(image)
        pred_mask = torch.sigmoid(output).cpu().squeeze().numpy()

    # POST-PROCESSING
    # Apply optimized threshold
    pred_mask_bin = (pred_mask > 0.8).astype(np.uint8)
    # Apply remove_small_objects
    pred_mask_bin = morph.remove_small_objects(pred_mask_bin.astype(bool), min_size=30).astype(np.uint8)
    # Apply morphological closing
    pred_mask_bin = morph.closing(pred_mask_bin, morph.disk(1))

    # Save predicted mask
    pred_mask_img = Image.fromarray(pred_mask_bin * 255)
    pred_mask_img.save(os.path.join(output_masks_dir, img_name))